# FinanceBench-RAG

Grounded question-answering over SEC 10-K filings, benchmarked on the FinanceBench dataset.

Notebook structure follows [SPEC.md §11](../SPEC.md):

- **Setup** - imports, env, Nebius client, dataset load/filter, shared helpers.
- **Task 1** - Naive generation (no retrieval).
- **Task 2** - RAG theory write-up.
- **Task 3** - Embed documents.
- **Task 4** - `answer_with_rag` pipeline.
- **Task 5** - Run & compare.
- **Task 6** - Evaluation (correctness + faithfulness + page-hit@k).
- **Task 7** - Improvement cycles.
- **Bonus** - Multi-scale chunking.

In [1]:
# Install dependencies. Safe to re-run - pip will skip what's already satisfied.
# Mirrors requirements.txt; kept here so the notebook runs end-to-end without external setup.
%pip install -q \
    python-dotenv \
    openai \
    pandas \
    openpyxl \
    datasets \
    huggingface_hub \
    tqdm \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    cryptography \
    sentence-transformers \
    ragas \
    requests

Note: you may need to restart the kernel to use updated packages.


## Setup

Imports, env loading, Nebius client, dataset load + filter, and shared helpers used across tasks.
Only the pieces needed by tasks already implemented are wired up - later setup cells will be added in their own task sections as we grow the notebook.

In [2]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

load_dotenv(REPO_ROOT / ".env")

NEBIUS_API_KEY = os.environ["NEBIUS_API_KEY"]
NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1"

GENERATION_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
JUDGE_MODEL = "deepseek-ai/DeepSeek-V3-0324"

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print(f"Nebius client ready. Generation model: {GENERATION_MODEL}")

Nebius client ready. Generation model: meta-llama/Llama-3.3-70B-Instruct


/opt/homebrew/Caskroom/miniconda/base/envs/nebius/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def chat(
    messages: list[dict],
    model: str = GENERATION_MODEL,
    temperature: float = 0.0,
    max_tokens: int = 512,
    max_retries: int = 3,
) -> str:
    """Thin wrapper over Nebius chat completions with simple retry."""
    last_err: Exception | None = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last_err = exc
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Nebius chat failed after {max_retries} retries: {last_err}")

smoke = chat(
    [{"role": "user", "content": "Reply with the single word: ready"}],
    max_tokens=8,
)
print(f"Smoke test → {smoke!r}")

Smoke test → 'ready'


### Dataset load, filter, and URL fix

Per [SPEC.md §3](../SPEC.md#3-dataset-contract):

1. Drop `question_type == "metrics-generated"` rows.
2. Replace dead `doc_link` URLs with the mirror repo: [patronus-ai/financebench](https://github.com/patronus-ai/financebench/tree/main/pdfs) (raw base: `https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs`).
3. `evidence_page_num` stays 0-indexed (we'll re-use this in Task 3 onward).

We hold the filtered DataFrame in `df` for the rest of the notebook.

In [4]:
from datasets import load_dataset

FINANCEBENCH_MIRROR = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"

raw = load_dataset("PatronusAI/financebench", split="train")
df_full = raw.to_pandas()

df = df_full[df_full["question_type"] != "metrics-generated"].copy()
df["doc_link"] = df["doc_name"].apply(lambda n: f"{FINANCEBENCH_MIRROR}/{n}.pdf")
df = df.sort_values("financebench_id").reset_index(drop=True)

print(f"Full rows: {len(df_full)}")
print(f"After filter: {len(df)}")
print("Question types:", df["question_type"].value_counts().to_dict())
df.head(3)

Full rows: 150
After filter: 100
Question types: {'domain-relevant': 50, 'novel-generated': 50}


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Trade accounts receivable, net of doubtful acc...",OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Information Technology,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
1,financebench_id_00070,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",Accounts receivable+Income tax receivable+Unbi...,OPEN_SOURCE,[{'evidence_text': 'American Water Works Compa...,Utilities,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
2,financebench_id_00080,Paypal,PAYPAL_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"Accounts receivable, net+Loans and interest re...",OPEN_SOURCE,"[{'evidence_text': 'PayPal Holdings, Inc. CONS...",Financials,10k,2022,https://raw.githubusercontent.com/patronus-ai/...


## Task 1 - Naive Generation

**Goal.** Hit Llama-3.3-70B-Instruct with each question directly - no retrieval, no context - and record what the model says.

**Sample.** First 5 questions of each `question_type` (sorted by `financebench_id`): 5 *domain-relevant* + 5 *novel-generated* = 10 questions.

**Verdict.** The assignment requires **manual judgment** - verdicts are filled in by hand after eyeballing each `(question, naive_answer, ground_truth)` row against the 4-way label space `{correct, partially correct, wrong, refused}`. The notebook writes the xlsx with the `verdict` column blank, ready to be filled in.

In [5]:
TASK1_N_PER_TYPE = 5

task1_df = (
    df.sort_values("financebench_id")
      .groupby("question_type", group_keys=False)
      .head(TASK1_N_PER_TYPE)
      .reset_index(drop=True)
)

print(f"Task 1 sample: {len(task1_df)} questions")
print(task1_df["question_type"].value_counts().to_dict())
task1_df[["financebench_id", "question_type", "question"]]

Task 1 sample: 10 questions
{'domain-relevant': 5, 'novel-generated': 5}


,financebench_id,question_type,question
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...


In [6]:
NAIVE_SYSTEM_PROMPT = (
    "You are a financial analyst answering questions about public companies. "
    "Answer concisely and directly. If you do not have enough information to answer, "
    "say so explicitly rather than guessing."
)

def naive_answer(question: str) -> str:
    return chat(
        [
            {"role": "system", "content": NAIVE_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        model=GENERATION_MODEL,
        temperature=0.0,
        max_tokens=400,
    )

naive_answers: list[str] = []
for q in tqdm(task1_df["question"].tolist(), desc="Naive generation"):
    naive_answers.append(naive_answer(q))

task1_df["naive_answer"] = naive_answers
task1_df[["financebench_id", "question_type", "question", "naive_answer"]].head()

Naive generation: 100%|██████████| 10/10 [00:30<00:00,  3.01s/it]


,financebench_id,question_type,question,naive_answer
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company's ..."
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"Based on American Water Works' FY2022 data, I ..."
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,"Based on PayPal's FY2022 data, the company's c..."
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for JP...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,"Yes, based on FY 2022 data, Verizon is conside..."


In [7]:
from datetime import datetime

task1_out = task1_df.rename(columns={"answer": "ground_truth"}).copy()
task1_out["verdict"] = ""  # filled in manually after review (see Discussion below)

task1_out = task1_out[
    [
        "financebench_id",
        "question_type",
        "question",
        "naive_answer",
        "ground_truth",
        "verdict",
    ]
]

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
task1_path = ARTIFACTS_DIR / f"assignment2_naive_generation_{_ts}.xlsx"
task1_out.to_excel(task1_path, index=False)
print(f"Wrote {task1_path.relative_to(REPO_ROOT)}  ({len(task1_out)} rows)")
print("Fill in the 'verdict' column, then save a copy named assignment2_naive_generation.xlsx for submission.")
task1_out

Wrote artifacts/assignment2_naive_generation_20260427_031426.xlsx  (10 rows)
Fill in the 'verdict' column, then save a copy named assignment2_naive_generation.xlsx for submission.


,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company's ...",Yes. Corning had a positive working capital am...,
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"Based on American Water Works' FY2022 data, I ...","No, American Water Works had negative working ...",
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,"Based on PayPal's FY2022 data, the company's c...",Yes. Paypal has a positive working capital of ...,
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for JP...,"Since JPM is a financial institution, gross ma...",
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,"Yes, based on FY 2022 data, Verizon is conside...",Yes. Verizon's capital intensity ratio was app...,
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,I don't have the most current information on P...,77.78,
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't have enough information to answer that...,"Yes, there was a decline of ~42% between FY202...",
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,I don't have access to JPM's 2021 Q1 financial...,Corporate. Its net revenue was -$473 million.,
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,I don't have the specific information on Pfize...,"Yes, change in PPNE was positive year over year",
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,I don't have the specific information on MGM's...,Las Vegas resorts contributed ~90% of company ...,


### Task 1 - Discussion

**Verdict distribution (manually judged, 4-way label space `{correct, partially correct, wrong, refused}`):**

| Verdict           | domain-relevant | novel-generated | Total |
|-------------------|-----------------|-----------------|-------|
| correct           | 2               | 0               | 2     |
| partially correct | 1               | 0               | 1     |
| wrong             | 1               | 0               | 1     |
| refused           | 1               | 5               | 6     |
| **Total**         | **5**           | **5**           | **10**|

**1. When did the model refuse to answer? Why do you think it refused?**

The model refused on 6 of 10 questions: 1 domain-relevant (American Water Works working capital) and all 5 novel-generated questions.

The refusals share the same shape: the model says it lacks specific company financials for the requested fiscal period and redirects the user to the company's filings or investor pages.

The novel-generated questions ask for exact figures or specific segment / region rankings (e.g., "How much does Pfizer expect to pay to spin off Upjohn in USD million?", "Which of JPM's business segments had the lowest net revenue in 2021 Q1?"), which the model cannot produce from parametric memory alone, so refusal is the safe choice.

The American Water Works refusal is the same pattern: a numeric balance sheet question on a less prominent issuer that the model has not memorized precisely.

Refusals correlate strongly with question specificity (exact numbers, narrow time windows, lesser-known issuers) rather than with `question_type` per se.

**2. When did the model answer confidently?**

The model answered confidently on 4 of 5 domain-relevant questions, all of which had a binary or qualitative shape (yes/no on working capital, yes/no on capital intensity, "is gross margin relevant for a bank") rather than asking for a precise figure.

Of those 4 confident answers:

- **2 correct:** JPM gross margin not relevant (correctly identified as a financial institution); Verizon capital intensive (correct yes/no with reasonable supporting reasoning on network infrastructure).
- **1 partially correct:** Corning working capital - right direction (positive) but wrong magnitude, because the model used full current assets / liabilities ($3.7B) instead of operating-only items ($831M ground truth).
- **1 wrong:** PayPal working capital - opposite sign vs. ground truth (model said negative -$4.5B, ground truth is positive $1.6B).

Confidence tracked with question shape: when the answer space is small and the issuer is well known, the model produces a fluent answer even when the supporting numbers are hallucinated.

This is the dangerous failure mode: a refusal is visible and easy to triage, but a confident answer paired with hallucinated supporting numbers looks authoritative and slips past unless someone checks the filing. RAG is built for exactly this case - grounding the supporting numbers in retrieved 10-K text turns a confident-sounding answer into a verifiable one.

**3. Are there patterns related to `question_type`?**

Yes, the split is sharp.

Domain-relevant questions (curated by FinanceBench) skew toward qualitative or binary judgments on prominent large-cap issuers; the model engaged on 4/5 with mixed accuracy.

Novel-generated questions skew toward precise figures, narrow time windows (single quarters, year-over-year deltas), and specific segment or region rankings; the model refused on 5/5.

Two failure modes are worth flagging for the later RAG comparison: (a) silent hallucination on confident answers (PayPal numbers, Corning magnitude) where the model produces plausible-looking figures that are wrong, and (b) blanket refusal on any question requiring an exact number it has not memorized.

RAG should help with both: grounding answers in retrieved 10-K text should reduce hallucination on the confident answers, and supplying the relevant filing should let the model attempt the questions it currently refuses.


## Task 2 - RAG Reminder

A simple RAG pipeline has three components: **indexing** (documents into a searchable vector store), **retrieval** (query into top-k relevant chunks), and **generation** (query + chunks into a grounded answer). For each component, three questions: what does it contribute, where can it fail, and does it run once offline or per query.

### Indexing

**1. Contribution.** Indexing turns a static document corpus into something queryable by meaning rather than keywords. Typically PDFs are loaded, split into chunks (here, 1000 chars with 150 overlap), embedded with a sentence-transformer model (we use `BAAI/bge-small-en-v1.5`), and stored in a vector index (FAISS in our case) alongside source metadata such as `doc_name` and `page_number`.

**2. Failure modes.** Bad chunk boundaries are the most common failure - e.g., splitting a balance-sheet row so the number survives but its label "Total current liabilities" is lost. Embedding model mismatch with the domain (a general-web sentence-transformer may not separate fiscal-year variants of the same financial metric well) and metadata mistakes (1-indexed vs. 0-indexed page numbers) also break things downstream.

**3. When it runs.** **Offline, once.** The index is built up front, persisted to disk, and reused across many queries. It is only rebuilt when the corpus or the chunking / embedding strategy changes.

### Retrieval

**1. Contribution.** Retrieval embeds the user question with the *same* model used at index time, then pulls the top-k nearest chunks from the vector index (FAISS in our case) using the index's similarity metric. This narrows the corpus from thousands of chunks down to a few that fit in the generator's context window, and it decides whether the right evidence is even available to the generator.

**2. Failure modes.** A recall miss - the evidence page is not in the top-k because k is too small or the question's wording is semantically far from the chunk's. The opposite also bites: top-k surfaces plausible-but-wrong chunks that the generator anchors to, e.g., FY2021 chunks for an FY2022 question, or a Merck filing chunk for a Pfizer question because both mention Upjohn-style spinoffs.

**3. When it runs.** **Online, per query.** Each user question triggers a fresh embed-and-search round trip against the index.

### Generation

**1. Contribution.** Generation takes the user query plus the retrieved chunks (formatted with clear separators and source metadata such as `doc_name`) and asks an LLM (we use `Llama-3.3-70B-Instruct`) to produce an answer grounded in those chunks rather than its parametric memory. The prompt also instructs the model to refuse when the chunks do not support an answer, which is what protects against the hallucination failure mode we saw in Task 1.

**2. Failure modes.** Ignoring the context and falling back to memorized knowledge (a faithfulness failure - the answer "looks right" but is not actually supported by the chunks). Misreading the chunks (mixing up two fiscal-year columns in the same chunk, picking the wrong row from a table). Miscalibrated refusals - refusing when the answer is actually present (over-cautious) or answering when the chunks are irrelevant (under-cautious).

**3. When it runs.** **Online, per query.** Each question gets its own generation call with its own freshly retrieved context.


## Task 3 - Embed Documents

Build the searchable index that the RAG pipeline retrieves from. Three steps: download the PDFs referenced by the filtered dataset, load and chunk them with page-level metadata, then embed and persist to a FAISS vector store. The index is built once and cached on disk so later tasks (and re-runs) load it instead of rebuilding.

Baseline configuration (per [SPEC.md §4.1](../SPEC.md#41-indexing-offline-once-per-chunk-configuration)): `chunk_size=1000`, `chunk_overlap=150`, `BAAI/bge-small-en-v1.5` embeddings. Metadata kept on every chunk: `doc_name`, `company`, `doc_period`, `page_number` (0-indexed, matching the dataset's `evidence_page_num` convention).

In [8]:
# Download the PDFs referenced by the filtered dataset. Idempotent: skips files already on disk.
import requests

PDF_DIR = REPO_ROOT / "data" / "pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

doc_names = sorted(df["doc_name"].unique())
to_download = [n for n in doc_names if not (PDF_DIR / f"{n}.pdf").exists()]

print(f"Unique docs in filtered dataset: {len(doc_names)}")
print(f"Already cached: {len(doc_names) - len(to_download)}; to download: {len(to_download)}")

for name in tqdm(to_download, desc="Downloading PDFs"):
    url = f"{FINANCEBENCH_MIRROR}/{name}.pdf"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    (PDF_DIR / f"{name}.pdf").write_bytes(resp.content)

local = sorted(p.name for p in PDF_DIR.glob("*.pdf"))
print(f"Local PDFs after download: {len(local)} (expected {len(doc_names)})")
assert len(local) == len(doc_names), "PDF count mismatch - investigate before building the index."


Unique docs in filtered dataset: 42
Already cached: 42; to download: 0


Local PDFs after download: 42 (expected 42)


In [9]:
# Build (or load) the FAISS index. Loader -> page-level metadata -> recursive chunking -> bge embeddings -> FAISS.
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
INDEX_DIR = REPO_ROOT / "indices" / f"faiss_chunk{CHUNK_SIZE}"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

if INDEX_DIR.exists():
    print(f"Loading cached index from {INDEX_DIR.relative_to(REPO_ROOT)}")
    vectorstore = FAISS.load_local(
        str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
    )
else:
    print(f"Building FAISS index at {INDEX_DIR.relative_to(REPO_ROOT)}")
    doc_meta = (
        df.drop_duplicates("doc_name")
          .set_index("doc_name")[["company", "doc_period"]]
          .to_dict("index")
    )

    page_docs = []
    for name in tqdm(doc_names, desc="Loading PDFs"):
        loader = PyPDFLoader(str(PDF_DIR / f"{name}.pdf"))
        for i, page in enumerate(loader.load()):
            page.metadata = {
                "doc_name": name,
                "company": doc_meta[name]["company"],
                "doc_period": doc_meta[name]["doc_period"],
                "page_number": i,  # 0-indexed, matches dataset's evidence_page_num
            }
            page_docs.append(page)
    print(f"Loaded {len(page_docs)} pages from {len(doc_names)} PDFs.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )
    chunks = splitter.split_documents(page_docs)
    print(f"Split into {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}).")

    vectorstore = FAISS.from_documents(chunks, embeddings)
    INDEX_DIR.parent.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))
    print(f"Saved index to {INDEX_DIR.relative_to(REPO_ROOT)}")

print(f"Index size: {vectorstore.index.ntotal} vectors")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7890.51it/s]


Building FAISS index at indices/faiss_chunk1000


Loading PDFs: 100%|██████████| 42/42 [03:54<00:00,  5.59s/it]


Loaded 5448 pages from 42 PDFs.
Split into 24218 chunks (size=1000, overlap=150).
Saved index to indices/faiss_chunk1000
Index size: 24218 vectors


In [12]:
# Spot-check retrieval on a few questions: do the top-k chunks land on or near the labeled evidence pages?
import ast

def _evidence_pages(row) -> list[int]:
    """Return the set of evidence page numbers that belong to row['doc_name'].

    The dataset's schema permits cross-doc evidence (each evidence entry carries
    its own doc_name); empirically this filtered set never uses it, but we filter
    defensively to avoid page-number collisions across docs if that ever changes.
    """
    ev = row["evidence"]
    if isinstance(ev, str):
        ev = ast.literal_eval(ev)
    return sorted({
        int(e["evidence_page_num"])
        for e in ev
        if e.get("doc_name", row["doc_name"]) == row["doc_name"]
    })

SPOT_CHECK_IDS = [
    "financebench_id_00005",  # Corning working capital (domain-relevant)
    "financebench_id_00206",  # JPM gross margin (domain-relevant)
    "financebench_id_00283",  # Pfizer Upjohn spinoff (novel-generated)
]

for fid in SPOT_CHECK_IDS:
    row = df.loc[df["financebench_id"] == fid].iloc[0]
    ev_pages = _evidence_pages(row)
    print("=" * 88)
    print(f"{fid}  ({row['question_type']})  doc={row['doc_name']}  evidence_pages={ev_pages}")
    print(f"Q: {row['question']}")
    print("-" * 88)
    hits = vectorstore.similarity_search(row["question"], k=4)
    for i, d in enumerate(hits, 1):
        same_doc = d.metadata["doc_name"] == row["doc_name"]
        page_hit = d.metadata["page_number"] in ev_pages
        if same_doc and page_hit:
            flag = "HIT_doc_and_pg"
        elif same_doc:
            flag = "same_doc      "
        else:
            flag = "              "
        snippet = d.page_content.replace("\n", " ")[:160]
        print(f"  [{flag}] k={i}  {d.metadata['doc_name']}  p{d.metadata['page_number']}  | {snippet}")
    print()


financebench_id_00005  (domain-relevant)  doc=CORNING_2022_10K  evidence_pages=[59]
Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
----------------------------------------------------------------------------------------
  [same_doc      ] k=1  CORNING_2022_10K  p101  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since
  [same_doc      ] k=2  CORNING_2022_10K  p102  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since
  [              ] k=3  3M_2022_10K  p37  | not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled measures used by other companies. The Compa
  [              ] k

### Task 3 - Discussion

**Spot-check observations.** Three sample questions probed against the index, summarized below; per-row detail and the per-axis read follow. We use top-k = 4 here to match the Task 4 default `answer_with_rag(k=4)`; the brief leaves Task 3's k open.

| Question                       | Evidence doc          | Evidence page(s) | Right document? @ k=4   | Right page? @ k=4 | Top chunks close to evidence text? |
|--------------------------------|-----------------------|------------------|--------------------------|--------------------|-------------------------------------|
| Corning working capital        | `CORNING_2022_10K`    | 59               | 2/4                      | 0/4                | No, but k=4 is a 3M working-capital table (right shape, wrong issuer) |
| JPM gross margin               | `JPMORGAN_2022_10K`   | 2                | 0/4 (1/4 right issuer, wrong filing) | 0/4    | No (MD&A boilerplate)               |
| Pfizer Upjohn future payment   | `Pfizer_2023Q2_10Q`   | 40               | 0/4 (4/4 right issuer, wrong filing) | 0/4    | No (2020-era Upjohn mechanics)      |

Read on each axis:

- **Right document?** Mixed to poor. Corning landed 2/4 chunks inside `CORNING_2022_10K`; JPM hit 0/4 inside `JPMORGAN_2022_10K` (k=2 is `JPMORGAN_2022Q2_10Q`, the right issuer but the wrong filing); Pfizer hit 0/4 inside the evidence doc `Pfizer_2023Q2_10Q`, although all 4 of its chunks came from `PFIZER_2021_10K` (same issuer, wrong filing).
- **Close to the evidence text?** Not really. None of the top-4 chunks cover the content the evidence pages actually contain (Corning's consolidated balance sheet, JPM's gross-margin discussion, Pfizer's future spin-off payment commitment). The closest near-miss is the Corning case: k=4 surfaces 3M's working-capital table (current assets minus current liabilities), which is structurally what we wanted but from the wrong issuer. JPM's chunks are MD&A boilerplate (a BestBuy disclaimer at k=1, a JPM 10-Q metrics table at k=2, AMD and Boeing accounting-estimate language at k=3 and k=4); Pfizer's chunks are 2020-era Upjohn spin-off mechanics rather than the forward-looking number the question asks for.
- **Right page number?** Zero `HIT_doc_and_pg` across the three questions. `page_hit@4` on this small sample is 0/3.

Per-row detail:

- **Corning** (evidence p59, consolidated balance sheet). k=1 and k=2 stayed inside `CORNING_2022_10K` but landed on pages 101 and 102 (the HSG consolidation note, near-identical lead text on both pages), not p59. k=3 is `3M_2022_10K` p37, non-GAAP measures definition language. k=4 is `3M_2023Q2_10Q` p70, an actual working-capital table (current assets, current liabilities, working-capital line) for 3M. The embedder pulled the right *shape* of evidence into the top-4, but from the wrong company; meanwhile the same-doc chunks it found were Corning footnote material rather than the balance sheet.
- **JPM gross margin** (evidence p2): top-1 is a BestBuy 10-K disclaimer paragraph; k=2 is `JPMORGAN_2022Q2_10Q` (right issuer, wrong filing: a 10-Q rather than the labeled 10-K); k=3 and k=4 are AMD and Boeing MD&A boilerplate. The question's "gross margins... fluctuating" phrasing maps to generic risk-factor language that exists in many 10-Ks, and "JPM" is too short to dominate the embedding.
- **Pfizer Upjohn** (evidence p40 in `Pfizer_2023Q2_10Q`, future spin-off cost). All 4 chunks come from `PFIZER_2021_10K`. k=1, k=2, k=4 discuss the Upjohn spin-off as past history (a 2020 cash-flow note, the "Transforming to a More Focused Company" narrative, and the VTRS NASDAQ listing detail). k=3 is a Pfizer revenue / alliance / biosimilars table where the Consumer Healthcare line shows post-spin-off zeros. None contain the forward-looking payment commitment the question asks for. The embedder picked the older filing that talks about Upjohn the most, missing the newer 10-Q.

**Useful weak baseline.** Three distinct failure shapes appeared, each mapping cleanly to a Task 7 experiment:

- **Right doc, wrong page** (Corning). Retrieval found the correct issuer's 10-K but landed on a footnote rather than the balance sheet. *Lever:* increase `k` (E1) so the relevant page has more chances to appear in the top-k.
- **Issuer confusion under generic question wording** (JPM). The question's vocabulary mapped to MD&A boilerplate that exists across many 10-Ks; the issuer signal was too weak to dominate. *Lever:* a cross-encoder reranker (E3) or a stricter prompt (E4) to penalize chunks that are not from the named issuer.
- **Filing-year confusion** (Pfizer). The embedder picked the older filing where the entity is discussed most often, missing the newer filing that holds the forward-looking number. *Levers:* smaller chunks (E2) so the embedder commits to less content per chunk, plus metadata filtering by `doc_period` if needed.

---

**Why these settings.** Baseline `chunk_size=1000` with `chunk_overlap=150` is a balance: small enough that a single chunk is roughly one topic in a 10-K (a paragraph or a small table block) and the embedding stays sharp, large enough that a balance-sheet line and its label usually survive in the same chunk. The 150-char overlap reduces the chance that a fact gets cut at the seam. Bonus task explores 300 and 2000 to see whether this baseline is actually best for FinanceBench.

**Why `BAAI/bge-small-en-v1.5`.** Small (384-dim, ~130 MB), strong on English retrieval benchmarks, and CPU-friendly so the index builds in single-digit minutes on a laptop. Good fit for a baseline; if we see the embedder confuse fiscal-year variants of the same metric we can swap to a larger BGE or a finance-specific encoder later.

**Why FAISS.** Local, file-backed, no service to run. `save_local` produces a directory we can re-load in any later cell or experiment, which is exactly the persistence story the spec calls for.

**Page metadata.** We attach our own `page_number` field set to the loader's enumeration index, deliberately overwriting `PyPDFLoader`'s default `page` field. The dataset's `evidence_page_num` is 0-indexed; we keep our metadata 0-indexed too so `page_hit@k` (Task 6) is a direct equality check with no off-by-one.
